# 06 · Model artifacts: publishing, prefetching, and promotion

**Story so far:** the corpus is processed ([03](./03-processing-at-scale.ipynb)), the
pipeline survives failure ([05](./05-resilience-and-recovery.ipynb)), and iteration is
cheap ([04](./04-caching-and-reproducibility.ipynb)). Review Radar now needs a **model** —
and it is not trained here. The weights come out of your own training stack and land in
object storage. The platform's job is to give those bytes an *identity*: a name and version
you can pin, metadata that travels with them, lineage back to what produced them, and a
promotion path into production.

Flyte 2 has a first-class **artifact service** for exactly this: named, versioned, typed
values with metadata, cards, and provenance — plus triggers that fire when a new version
lands.

> **Coming from v1?** Every v1 artifact concept has a v2 counterpart, though the spelling
> changed: `Artifact(name=...)` + `Annotated[FlyteFile, ...]` became
> `flyte.artifacts.new(value, Metadata(...))`, `ModelCard` became `artifacts.Card`, and
> `OnArtifact` is still called `OnArtifact`. The full mapping is in
> [09](./09-migration-v1-to-v2.ipynb) §1.

**Learning goals**

1. Publish a model to the artifact service with `flyte.artifacts.new` + `Metadata`
2. Attach a **card** and searchable `attrs`, and read them back
3. Adopt weights trained **outside** Flyte, keeping a reference to where they came from
4. Prefetch HuggingFace weights into your own bucket with `flyte.prefetch.hf_model`
5. Pin, promote, and roll back the version an app serves
6. Fire a pipeline when a new version lands, with `flyte.OnArtifact`

> **Version note:** the artifact service and `OnArtifact` require **flyte 2.6+**
> (this workshop pins 2.6.13). On 2.5.x neither exists.

In [20]:
import flyte
import flyte.io

from workshop_config import WS      # client-side values only (authoring rule 1)

flyte.init_from_config()

## 1. What an artifact is in v2

An **artifact** is a named, versioned, typed value in the artifact service. Only *offloaded
assets* can be artifacts — `flyte.io.File`, `Dir`, or `DataFrame`. Primitives, dataclasses
and pydantic models are rejected, because an artifact is a pointer to bytes, not the bytes
themselves.

Every artifact carries:

| | |
|---|---|
| `name` + `version` | its identity; version defaults to random, so pass one you control |
| `kind` | `"model"`, `"data"`, or `"generic"` — stored under the reserved `flyte.io/kind` attr |
| `attrs` | free-form `str → str` metadata, and **`listall` can filter on it** |
| `card` | a rendered report (html/md/json/png/…) attached to the version — v1's `ModelCard` |
| `description` | human-readable text |
| `source` | provenance: the producing action, or an `external_ref` you supply |

Two ways to publish, and they differ in **who does the writing**:

| | How it publishes | |
|---|---|---|
| `Artifact.create(value, name=..., ...)` | issues `CreateArtifactRequest` **itself** | works wherever the artifact service is reachable — a task, a script, your laptop. Use `await Artifact.create.aio(...)` inside an `async` task |
| `flyte.artifacts.new(value, Metadata(...))` returned from a task | **declares** the artifact on the task's output envelope and leaves registration to the control plane | the tidier idiom, and the successor to v1's `Annotated[FlyteFile, my_artifact]` — but it publishes nothing on a deployment that doesn't act on declarations |


And to read them back:

```python
from flyte.remote import Artifact

art = Artifact.get("review-radar-sentiment")             # newest version
art = Artifact.get("review-radar-sentiment", "v3")       # exact version
model_file = await art.to_python()                       # back to a flyte.io.File

for a in Artifact.listall(kind="model", attrs={"stage": "prod"}):
    print(a.tracker, a.version, a.source)   # org/project/domain/name@version
```

> **Sync here, `.aio` inside a task — with one exception.** `Artifact.get` / `listall` /
> `list_names` / `create` / `delete` are *syncified*: call them plainly from a notebook cell
> or script, and use `await Artifact.get.aio(...)` inside an `async def` task body. Calling
> the blocking form inside a task raises a deadlock assertion rather than hanging.
>
> `Artifact.to_python()` and `coerce_to_literal()` are **not** syncified — they are plain
> coroutines, so they always need `await`, everywhere. Calling `art.to_python()` without it
> silently yields a coroutine object, and you find out one attribute access later:
> `AttributeError: 'coroutine' object has no attribute 'path'`.

Not everything has to become an artifact. Weights that already sit in a bucket can be
referenced with `File.from_existing_remote(uri)` and handed straight to an app. You publish
when you want the *identity*: a version to pin, metadata to search, and lineage to trace.

## 2. Publishing the model

Review Radar's weights are produced by your own training stack and land in object storage.
The **publish** task adopts them: it checks they load, attaches the metadata you will want
six months from now, and registers them as a named artifact version.

First we need weights in the bucket. In the engagement these come from your training stack;
the cell below is a **stand-in** so the chapter runs end-to-end — delete it and set
`WEIGHTS_URI` to a real path when you adapt this.

In [21]:
# ── STAND-IN for your external training stack. Delete in the engagement. ──────────
# Produces a small sklearn sentiment model and leaves it in the object store, which is
# all we need: some weights sitting at a URI, produced outside the pipeline we care about.
ml_image = (
    flyte.Image.from_debian_base(name="workshop-model", python_version=(3, 12))
    .with_pip_packages("scikit-learn==1.5.2", "joblib==1.4.2")
)

external = flyte.TaskEnvironment(
    name="external_trainer",
    image=ml_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
)

POS = ["absolutely love this {p}", "solid {p}, works as advertised", "the {p} exceeded expectations"]
NEG = ["disappointed with the {p}", "terrible {p}, arrived broken", "the {p} stopped working, support ignored me"]
PRODUCTS = ["espresso machine", "trail shoes", "headphones", "air fryer"]


def make_training_data(n: int, seed: int) -> tuple:
    """Helper defined in a cell → ships with the pickled bundle (authoring rule 1)."""
    import random
    rng = random.Random(seed)
    texts, labels = [], []
    for _ in range(n):
        label = rng.random() < 0.5
        tpl = rng.choice(POS if label else NEG)
        texts.append(tpl.format(p=rng.choice(PRODUCTS)))
        labels.append(int(label))
    return texts, labels


@external.task
async def emit_weights(n_examples: int = 2000, c: float = 1.0, seed: int = 7) -> flyte.io.File:
    import joblib
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline

    texts, labels = make_training_data(n_examples, seed)
    model = make_pipeline(TfidfVectorizer(), LogisticRegression(C=c, max_iter=1000))
    model.fit(texts, labels)
    joblib.dump(model, "sentiment.joblib")
    return await flyte.io.File.from_local("sentiment.joblib")


run = flyte.run(emit_weights)
print(run.url)
run.wait()

# The URI the "external trainer" left behind. Point this at a real bucket path instead.
# `run.outputs()` is an ActionOutputs tuple, so index the first (only) output.
WEIGHTS_URI = run.outputs()[0].path
print("weights at:", WEIGHTS_URI)

> Building 1 image...

> Building image workshop-model for environment external_trainer

✓ Built image for environment external_trainer: 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:workshop-model-c93f885d07aa65c953aca82a79efca02

Output()

https://demo.hosted.unionai.cloud/v2/domain/development/project/davide/runs/u45cg5z2cgx5hshg8gqw


Run 'u45cg5z2cgx5hshg8gqw' completed successfully.

weights at: s3://union-oc-production-demo-raw/uc/demo/davide/development/u45cg5z2cgx5hshg8gqw/a0/u45cg5z2cgx5hshg8gqw-a0-0/0eb120e10f706bbbdd47c3c0aa90a596/sentiment.joblib


In [22]:
# ── Publish the weights as a named, versioned artifact, with a model card. ────────
import flyte.artifacts as artifacts
from flyte.remote import Artifact

model_registry = flyte.TaskEnvironment(
    name="model_registry",
    image=ml_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
)

MODEL_NAME = "review-radar-sentiment"

# Versions are CHOSEN, not automatic. Omit `version=` and the service assigns a random
# uuid; pass one and you control it. Re-publishing the SAME name@version does not create
# a new version — bump this to "v2", "v3", ... to publish a new one.
MODEL_VERSION = "v2"


def render_model_card(name, version, estimator, trained_by, eval_accuracy, uri, notes):
    """The v1 ModelCard, as ordinary HTML. Defined in a cell so it ships with the
    pickled bundle (authoring rule 1)."""
    return f"""<h1>{name} <code>{version}</code></h1>
<p>{notes}</p>
<h2>At a glance</h2>
<table>
  <tr><th align="left">Estimator</th><td><code>{estimator}</code></td></tr>
  <tr><th align="left">Trained by</th><td>{trained_by}</td></tr>
  <tr><th align="left">Eval accuracy</th><td>{eval_accuracy}</td></tr>
  <tr><th align="left">Weights</th><td><code>{uri}</code></td></tr>
</table>
<h2>Intended use</h2>
<p>Sentiment scoring of product reviews for Review Radar triage. Not for
moderation decisions without a human in the loop.</p>
<h2>Limitations</h2>
<ul>
  <li>English-language reviews only.</li>
  <li>Trained on the product categories present in the source corpus; expect
      degradation outside them.</li>
  <li>No calibration guarantees — treat scores as ordinal, not as probabilities.</li>
</ul>
"""


@model_registry.task
async def publish_model(uri: str, version: str, trained_by: str,
                        eval_accuracy: str, notes: str = "") -> flyte.io.File:
    """Adopt externally-trained weights as a versioned artifact, with a card.

    `from_existing_remote` references the bytes where they already live — nothing is
    copied. `Artifact.create` then registers that File as a version of MODEL_NAME.

    Note `.aio`: these remote calls are syncified, so inside an `async` task body you
    await the `.aio` form. Calling the blocking form here raises a deadlock assertion.
    """
    import joblib

    f = flyte.io.File.from_existing_remote(uri)
    local = await f.download()

    model = joblib.load(local)          # fail here, not at serving time
    estimator = type(model).__name__

    # Upload the card FIRST — it returns a Card holding the uploaded URI. Inside a task
    # this lands under the action's output path with the right Content-Type, so the
    # console renders it inline instead of downloading it.
    card = await artifacts.Card.create_from.aio(
        content=render_model_card(MODEL_NAME, version, estimator, trained_by,
                                  eval_accuracy, uri, notes),
        format="html",
        card_type="model",
    )

    art = await Artifact.create.aio(
        f,
        name=MODEL_NAME,
        version=version,
        description=notes or "Review Radar sentiment classifier",
        kind="model",                      # -> flyte.io/kind, filterable in listall
        card=card,                         # <- the v1 ModelCard
        attrs={                            # free-form, searchable metadata
            "trained_by": trained_by,
            "eval_accuracy": eval_accuracy,
            "source_uri": uri,             # where the weights came from
            "estimator": estimator,
            "stage": "candidate",          # promote by republishing with stage=prod
        },
        # REQUIRED when calling create() from inside a task, not just a nicety:
        # without external_ref the SDK builds a task-action source whose run identifier
        # has empty org/project/domain, which the server rejects with
        #   spec.source.task_action.action.run.org: must be at least 1 characters
        # Passing external_ref takes a different branch and records the upstream URI as
        # the artifact's provenance instead. See the note below.
        external_ref=uri,
    )
    print(f"published {art.tracker}")
    print(f"  card: {card.uri}")
    print(f"  url:  {art.url}")

    # Return the File (not the Artifact) so chapter 07's RunOutput wiring still resolves
    # this task's output to a model file.
    return f


run = flyte.run(
    publish_model,
    uri=WEIGHTS_URI,
    version=MODEL_VERSION,
    trained_by="muon-training-stack",
    eval_accuracy="0.91",
    notes="baseline sentiment model",
)
print(run.url)
run.wait()

> Building 1 image...

> Building image workshop-model for environment model_registry

✓ Built image for environment model_registry: 356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo:workshop-model-c93f885d07aa65c953aca82a79efca02

Output()

https://demo.hosted.unionai.cloud/v2/domain/development/project/davide/runs/ul97lmxm2lfsgx9ssrjf


Run 'ul97lmxm2lfsgx9ssrjf' completed successfully.

The model is now a **version of `review-radar-sentiment`** in the artifact service, not just
a file at a path.

Note `version=MODEL_VERSION`. Version defaults to a random uuid, which is fine for
machine-generated artifacts and bad for anything a human has to pin — pass a version you
control, and bump it to publish a new one.


In [ ]:
# Verify the publish landed. If this raises or lists nothing, see the troubleshooting
# note below.
from flyte.remote import Artifact

art = Artifact.get(MODEL_NAME)                       # newest version; or (MODEL_NAME, "v1")
print("tracker:   ", art.tracker)                    # org/project/domain/name@version
print("kind:      ", art.kind)
print("source:    ", art.source)                     # external_ref: the upstream weights URI
print("created_by:", art.created_by)
print("url:       ", art.url)
print("card:      ", art.pb2.spec.info.card.uri or "(none)",
      f"[{art.pb2.spec.info.card.format}/{art.pb2.spec.info.card.type}]")

# to_python() is a plain coroutine — unlike get/listall/create/delete it is NOT
# syncified, so it always needs `await` (notebook cells allow top-level await).
model_file = await art.to_python()                   # back to a flyte.io.File
print("\nall versions of", MODEL_NAME)
for a in Artifact.listall(name=MODEL_NAME):
    print(f"  {a.version:20s} {a.source}")

### Cards: v1's `ModelCard`

A **card** is a rendered report attached to an artifact version — the model's story
travelling with it in the console. It is the direct successor to v1's `ModelCard`, and the
publish task above attaches one.

Two steps, and the first is easy to skip: a `Card` holds a **URI**, so the content has to be
uploaded before you can reference it. `Card.create_from` does both:

```python
card = await artifacts.Card.create_from.aio(
    content="<h1>my model</h1>…",     # or local_path=Path("eval-report.html")
    format="html",                    # html md json yaml csv tsv png jpg jpeg
    card_type="model",                # model | data | generic
)
art = await Artifact.create.aio(f, name=..., card=card, ...)
```

Called **inside a task**, it uploads under that action's output path with the correct
`Content-Type`, so the console renders the card inline rather than downloading it. Called
outside one, it goes through the control plane's upload endpoint. The object name is
content-addressed, so re-uploading identical bytes is idempotent and two concurrent cards of
the same type cannot clobber each other.

> **`card_type` is not `kind`.** `kind` says what the *artifact* is and drives
> `listall(kind=...)`; `card_type` says how the *card* renders. Set both — for older
> artifacts published before `flyte.io/kind` existed, `Artifact.kind` falls back to the
> card's type.

The natural producer is an evaluation task: score the candidate on a holdout set, render the
metrics to HTML, and publish the model with that card attached. `flyte.prefetch.hf_model`
does the same thing automatically, attaching the HuggingFace repo's README as the card.

> 💬 **Discuss:** what does the customer need to know about a model six months from now —
> training-data snapshot, eval scores, approver, upstream commit? Searchable facts belong in
> `attrs` (you can filter `listall` on them); anything a human needs to *read* belongs in the
> card.

## 3. Prefetching HuggingFace models

When the weights come from HuggingFace rather than your own trainer, `flyte.prefetch`
streams them **straight into your object store** — no laptop round trip, and no per-pod
download from the internet at serving time:

- **Faster starts:** the app mounts weights that are already local to your cloud
- **Reproducibility:** the model is versioned in your bucket, not pulled live from the Hub
- **Egress and offline control:** one download, then everything stays inside your network
- **Sharding:** pre-shard large models for multi-GPU tensor parallelism

It returns a run and registers the model under `artifact_name`, so a prefetched model is
addressable by `Artifact.get(...)` exactly like the one you published above.

In [ ]:
# Set a repo to run this. It launches a real prefetch run, and gated repos need an HF_TOKEN
# secret on the deployment (`flyte create secret HF_TOKEN`). Leave empty to skip.
WORKSHOP_HF_MODEL = ""     # e.g. "Qwen/Qwen3-0.6B"

if WORKSHOP_HF_MODEL:
    import flyte.prefetch

    run = flyte.prefetch.hf_model(
        repo=WORKSHOP_HF_MODEL,
        artifact_name="review-radar-llm",     # defaults to the repo name with '.' → '-'
        short_description="workshop LLM for chapter 07's vLLM app",
        hf_token_key="HF_TOKEN",              # the name of the Flyte secret, not the token
    )
    print(run.url)
    run.wait()
    print(run.outputs())
else:
    print("Set WORKSHOP_HF_MODEL to prefetch a model into this deployment's object store.")

The same thing from the CLI, which is what a CI job would call:

```bash
flyte prefetch hf-model Qwen/Qwen3-0.6B --wait
```

And for a model too large for one GPU, pre-shard it once instead of at every app start:

```python
from flyte.prefetch import ShardConfig, VLLMShardArgs

run = flyte.prefetch.hf_model(
    repo="meta-llama/Llama-2-70b-hf",
    shard_config=ShardConfig(engine="vllm", args=VLLMShardArgs(tensor_parallel_size=8)),
    hf_token_key="HF_TOKEN",
)
```

Sharding currently uses the `vllm` engine; the sharded output still loads from
`transformers`, `torch` or `sglang`. Chapter [07](./07-serving.ipynb) §3 serves one.

> **Prefetch is not the same as the app-side download.** `flyte.prefetch` moves weights
> *into your bucket*, once. `Parameter(download=True)` (next section) pulls them from your
> bucket *into the pod* before the app starts. Large models want both.

## 4. Pinning, promoting, rolling back

Two things can be pinned, and they answer different questions.

**By artifact name and version** — "which *model* is production serving?"

```python
art = Artifact.get("review-radar-sentiment", "v3")   # exact, immutable
art = Artifact.get("review-radar-sentiment")         # whatever is newest
```

**By run output** — "which *run* produced what this app is serving?" This is what an app
`Parameter` resolves at deploy time:

| Declaration | Resolves to | Use it for |
|---|---|---|
| `RunOutput(type="file", task_name="model_registry.publish_model")` | latest succeeded run of that task | "always serve the newest published model" |
| `RunOutput(..., task_auto_version="latest")` | latest succeeded run of the latest *deployed version* of that task | deployed (non-notebook) tasks |
| `RunOutput(..., task_version="abc123")` | latest succeeded run of one task version | pinning to a code version |
| `RunOutput(type="file", run_name="publish-model-x7k2")` | exactly that run | pinning one specific publish |

```python
from flyte.app import Parameter, RunOutput

Parameter(
    name="model",
    value=RunOutput(type="file", task_name="model_registry.publish_model"),
    download=True,              # pull it into the pod before startup
    env_var="MODEL_PATH",
)
```

And when there is no Flyte run at all — weights that simply exist in a bucket:

```python
Parameter(name="model", value="gs://muon-models/detector/v7/",
          type="directory", download=True, mount="/models")
```

**Promotion** is a metadata move: publish the blessed version again with `stage="prod"` in
`attrs`, and let consumers resolve `Artifact.listall(kind="model", attrs={"stage": "prod"})`.
**Rollback** is the same move pointed at the older version — no rebuild, because the artifact
already exists. Chapter [10](./10-event-driven-cd.ipynb) generalizes the idea to version
labels so the *caller* never changes either.

## 5. Firing on a new version: `flyte.OnArtifact`

The question everyone coming from v1 asks: *my launch plan used
`OnArtifact(trigger_on=my_model)` — what is that in v2?*

**It is still `OnArtifact`.** A trigger's `automation` takes `Cron`, `FixedRate`, or
`OnArtifact`, and `flyte.TriggeredArtifact` binds the artifact that fired it to a task
input — the same way `flyte.TriggerTime` binds the scheduled time:

```python
import flyte
from flyte.io import File

revalidate = flyte.Trigger(
    name="revalidate_on_new_model",
    automation=flyte.OnArtifact(name="review-radar-sentiment"),
    inputs={"model": flyte.TriggeredArtifact, "min_accuracy": 0.85},
)


@model_registry.task(triggers=[revalidate])
async def revalidate_model(model: File, min_accuracy: float) -> str:
    """Runs automatically whenever a new version of the model is published."""
    ...
```

`OnArtifact(name=...)` fires on **any** new version; add `version="v3"` to fire only when
precisely that version is created. The name is scoped to the task's project and domain.

Triggers need a deployed task, so this lives in
[`scripts/triggers_deploy.py`](./scripts/triggers_deploy.py), not a notebook cell
(authoring rule 3).

This closes the loop the workshop has been building toward: the training stack publishes a
version → the trigger fires validation → validation promotes by publishing with
`stage="prod"` → the serving app picks it up on its next deploy. Chapter
[10](./10-event-driven-cd.ipynb) covers the alternatives for when the producer lives
outside Flyte entirely (a webhook app, or CI calling the API).

**Story checkpoint:** the model has an identity — a name, a version, metadata, a card, and
lineage back to what produced it. Time to put it behind a live endpoint.

## Further reading

- Union docs: [Artifacts](https://www.union.ai/docs/v2/union/user-guide/) · [Prefetching models](https://www.union.ai/docs/v2/union/user-guide/serve-and-deploy-apps/prefetching-models/) · [Triggers](https://www.union.ai/docs/v2/union/user-guide/task-configuration/triggers/)
- Coming from v1 artifacts: [09](./09-migration-v1-to-v2.ipynb) §1
- Next: [07-serving](./07-serving.ipynb)